# IHGAMP - Step 4: Evaluation & Figure Generation

This notebook generates:
- **Table 2**: Primary metrics (AUC, AP, Brier, ECE) with bootstrap CIs
- **Table 3**: Clinical operating points (Youden-J, NPV≥0.95, PPV≥0.60)
- **Figure 2**: ROC and PR curves
- **Figure 3**: Calibration plots

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

from src.evaluation import (
    evaluate_with_calibration, compute_ece, calibrate_cv, select_operating_points
)

OUTPUT_DIR = Path('./outputs')
RESULTS_DIR = OUTPUT_DIR / 'results'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Load predictions
preds = pd.read_csv(OUTPUT_DIR / 'predictions.csv')
print(f'Loaded {len(preds)} predictions')

## Table 2: Primary Metrics

In [ ]:
all_metrics = []
all_ops = []

for split in ['val', 'test']:
    mask = preds['split'] == split
    y = preds.loc[mask, 'label'].values
    p = preds.loc[mask, 'pred'].values
    
    results = evaluate_with_calibration(
        y_true=y, y_prob=p,
        cohort_name=f'TCGA-{split.upper()}',
        n_bootstrap=2000
    )
    all_metrics.append(results['primary_metrics'])
    all_ops.append(results['operating_points'])

# Table 2
table2 = pd.concat(all_metrics, ignore_index=True)
table2['AUC (95% CI)'] = table2.apply(
    lambda r: f"{r['auc']:.3f} ({r['auc_ci_low']:.3f}-{r['auc_ci_high']:.3f})", axis=1
)
print('\n' + '='*80)
print('TABLE 2: Primary Performance Metrics')
print('='*80)
print(table2[['cohort', 'variant', 'n', 'AUC (95% CI)', 'ap', 'brier', 'ece_10bin']].to_string(index=False))
table2.to_csv(RESULTS_DIR / 'table2_primary_metrics.csv', index=False)

## Table 3: Operating Points

In [ ]:
table3 = pd.concat(all_ops, ignore_index=True)
print('\n' + '='*80)
print('TABLE 3: Clinical Operating Points')
print('='*80)
print(table3[['cohort', 'variant', 'rule', 'threshold', 'sensitivity', 'specificity', 'ppv', 'npv']].to_string(index=False))
table3.to_csv(RESULTS_DIR / 'table3_operating_points.csv', index=False)

## Figure 2: ROC and PR Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
colors = {'val': '#1f77b4', 'test': '#ff7f0e'}

# ROC
ax = axes[0]
for split in ['val', 'test']:
    mask = preds['split'] == split
    y, p = preds.loc[mask, 'label'].values, preds.loc[mask, 'pred'].values
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, color=colors[split], lw=2, label=f'{split.upper()} (AUC={auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')

# PR
ax = axes[1]
for split in ['val', 'test']:
    mask = preds['split'] == split
    y, p = preds.loc[mask, 'label'].values, preds.loc[mask, 'pred'].values
    prec, rec, _ = precision_recall_curve(y, p)
    ap = average_precision_score(y, p)
    ax.plot(rec, prec, color=colors[split], lw=2, label=f'{split.upper()} (AP={ap:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR Curves')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure2_roc_pr.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'figure2_roc_pr.pdf', bbox_inches='tight')
plt.show()

## Figure 3: Calibration Plots

In [ ]:
def plot_calibration(y, p, n_bins=10, ax=None, label='Model'):
    bins = np.linspace(0, 1, n_bins + 1)
    centers, accs = [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if mask.sum() > 0:
            centers.append((lo+hi)/2)
            accs.append(y[mask].mean())
    ax.plot(centers, accs, 'o-', label=label, markersize=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for idx, split in enumerate(['val', 'test']):
    ax = axes[idx]
    mask = preds['split'] == split
    y, p = preds.loc[mask, 'label'].values, preds.loc[mask, 'pred'].values
    plot_calibration(y, p, ax=ax, label='Uncalibrated')
    p_cal = calibrate_cv(y, p, method='platt')
    plot_calibration(y, p_cal, ax=ax, label='Platt (CV)')
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_title(f'{split.upper()} (ECE: {compute_ece(y,p):.3f} → {compute_ece(y,p_cal):.3f})')
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'figure3_calibration.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'figure3_calibration.pdf', bbox_inches='tight')
plt.show()

## LaTeX Export

In [ ]:
# Export Table 2 as LaTeX
latex_str = table2[['cohort', 'variant', 'n', 'AUC (95% CI)', 'ap', 'brier']].to_latex(index=False)
with open(RESULTS_DIR / 'table2_latex.tex', 'w') as f:
    f.write(latex_str)
print('LaTeX table saved to:', RESULTS_DIR / 'table2_latex.tex')